In [ ]:
! pip install -q gensim optuna lightgbm

In [ ]:
import pandas as pd
import numpy as np
import glob
from baseline import W2VSentenceEmbedder

random_state = 42

df_balanced = pd.read_csv("/kaggle/input/datasets/ul1oinasrchr/go-emotions/goemotions_balanced.csv")
print("строк:", len(df_balanced), "| колонки:", list(df_balanced.columns))
print("\nРаспределение меток:")
print(df_balanced["label"].value_counts())


In [ ]:
embedder = W2VSentenceEmbedder()

texts = df_balanced["text"].astype(str).tolist()

X = np.vstack(embedder(texts)).astype(np.float32)
y = df_balanced["label"].to_numpy()

print("X:", X.shape, "| y:", y.shape, "| dim:", embedder.vector_size)
print("классы:", np.unique(y))


In [ ]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import warnings

warnings.filterwarnings("ignore", message="X does not have valid feature names")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)


def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 400, step=100),
        "learning_rate":     trial.suggest_float("learning_rate", 1e-2, 3e-1, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 64),
        "max_depth":         trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    clf = LGBMClassifier(
        objective="multiclass",
        random_state=random_state,
        n_jobs=-1,
        class_weight="balanced",
        subsample_freq=1,    
        verbosity=-1,        
        **params,
    )
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=1)
    return scores.mean()


optuna.logging.set_verbosity(optuna.logging.INFO)

study = optuna.create_study(
    direction="maximize",
    study_name="lgbm_goemotions",
    storage="sqlite:///lgbm_goemotions.db",
    load_if_exists=True,
)

def log_cb(study, trial):
    print(f"[trial {trial.number}] f1_macro={trial.value:.4f} | best={study.best_value:.4f}", flush=True)

study.optimize(
    objective,
    n_trials=40,
    timeout=10 * 3600,        
    callbacks=[log_cb],
    show_progress_bar=False,
)

print("Лучший F1-macro (CV на train):", round(study.best_value, 3))
print("Лучшие параметры:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


In [ ]:
from sklearn.metrics import classification_report

best_clf = LGBMClassifier(
    objective="multiclass",
    random_state=random_state,
    n_jobs=-1,
    class_weight="balanced",
    subsample_freq=1,
    verbosity=-1,
    **study.best_params,
)
best_clf.fit(X_train, y_train)
y_pred = best_clf.predict(X_test)

print("classification_report (лучшие параметры Optuna, отложенная выборка):\n")
print(classification_report(y_test, y_pred, digits=3))


In [ ]:
import joblib

final_clf = LGBMClassifier(
    objective="multiclass",
    random_state=random_state,
    n_jobs=-1,
    class_weight="balanced",
    subsample_freq=1,
    verbosity=-1,
    **study.best_params,
)
final_clf.fit(X, y)

model_path = "lgbm_goemotions_optuna.joblib"
joblib.dump(
    {
        "model": final_clf,
        "classes": list(final_clf.classes_),
        "best_params": study.best_params,
        "best_cv_f1_macro": study.best_value,
        "embedder": "W2VSentenceEmbedder",
        "vector_size": int(X.shape[1]),
    },
    model_path,
)
print("Итоговая модель сохранена ->", model_path)

loaded = joblib.load(model_path)
print("Загружено, классы:", loaded["classes"])
